# 04 — Feature Engineering

This notebook creates useful click-prediction features from user, product, placement, and time information. Target-based features are built without using future clicks.

## Install required packages

In [1]:
%pip install pandas numpy joblib

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 150)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

current_dir = Path.cwd().resolve()
project_root = next(
    (path for path in [current_dir, *current_dir.parents] if (path / "data" / "processed").exists()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Could not find data/processed. Start Jupyter inside this project.")

processed_dir = project_root / "data" / "processed"
engineered_dir = processed_dir / "engineered"
engineered_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {engineered_dir}")

Output directory: /Users/namees.local/Desktop/ensamble_technique/data/processed/engineered


## Load and validate the chronological splits

In [3]:
train = pd.read_csv(processed_dir / "train.csv", parse_dates=["event_ts"])
validation = pd.read_csv(processed_dir / "validation.csv", parse_dates=["event_ts"])
test = pd.read_csv(processed_dir / "test.csv", parse_dates=["event_ts"])
splits = {"train": train, "validation": validation, "test": test}
target = "clicked"

for name, frame in splits.items():
    if frame[target].isna().any() or not set(frame[target].unique()).issubset({0, 1}):
        raise ValueError(f"{name} has an invalid clicked target.")
    if frame["impression_id"].duplicated().any():
        raise ValueError(f"{name} contains duplicate impression IDs.")

if not train["event_ts"].max() <= validation["event_ts"].min() <= validation["event_ts"].max() <= test["event_ts"].min():
    raise ValueError("Splits are not chronological.")

display(pd.DataFrame([
    {"split": name, "rows": len(frame), "columns": frame.shape[1], "click_rate": frame[target].mean()}
    for name, frame in splits.items()
]).set_index("split"))

,rows,columns,click_rate
split,,,
train,315000,22,0.0682
validation,67500,22,0.0701
test,67500,22,0.0706


## Feature plan

The notebook creates four groups of features:

1. **Time:** month, weekend, time of day, cyclic hour, and elapsed time.
2. **Product value:** final price, discount amount, discount flag, price position within category, rating quality, and popularity.
3. **Interactions:** combinations of context and user/product attributes.
4. **Historical behavior:** past impression counts and smoothed past CTR for users, products, categories, brands, pages, and user–category pairs.

## Learn training-only reference values

In [4]:
train = train.sort_values(["event_ts", "impression_id"]).reset_index(drop=True)
validation = validation.sort_values(["event_ts", "impression_id"]).reset_index(drop=True)
test = test.sort_values(["event_ts", "impression_id"]).reset_index(drop=True)

global_ctr = float(train[target].mean())
training_start = train["event_ts"].min()
category_price_median = train.groupby("category")["price_usd"].median()
category_rating_mean = train.groupby("category")["avg_rating"].mean()
price_bin_edges = np.unique(train["price_usd"].dropna().quantile([0, 0.25, 0.50, 0.75, 1]).to_numpy())
age_bin_edges = np.array([0, 24, 34, 44, 54, 64, 101])

reference_values = {
    "global_ctr": global_ctr,
    "training_start": str(training_start),
    "price_bin_edges": price_bin_edges.tolist(),
    "age_bin_edges": age_bin_edges.tolist(),
}
display(pd.Series(reference_values, name="training reference").to_frame())

,training reference
global_ctr,0.0682
training_start,2026-01-01 00:00:00
price_bin_edges,"[6.49, 21.95, 46.49, 111.99, 2064.95]"
age_bin_edges,"[0, 24, 34, 44, 54, 64, 101]"


## Create row-level time, product, and interaction features

In [5]:
def add_row_features(frame):
    result = frame.copy()

    # Time features available at impression time.
    result["event_month"] = result["event_ts"].dt.month.astype("int8")
    result["event_day_of_month"] = result["event_ts"].dt.day.astype("int8")
    result["event_weekday_number"] = result["event_ts"].dt.dayofweek.astype("int8")
    result["is_weekend"] = result["event_weekday_number"].isin([5, 6]).astype("int8")
    result["time_of_day"] = pd.cut(
        result["hour_of_day"], [-1, 5, 11, 16, 20, 23],
        labels=["overnight", "morning", "afternoon", "evening", "night"],
    ).astype("string")
    result["hour_sin"] = np.sin(2 * np.pi * result["hour_of_day"] / 24)
    result["hour_cos"] = np.cos(2 * np.pi * result["hour_of_day"] / 24)
    result["days_since_training_start"] = (result["event_ts"] - training_start).dt.total_seconds() / 86_400

    # Product value and popularity features.
    result["has_discount"] = result["discount_pct"].gt(0).astype("int8")
    result["discount_amount_usd"] = result["price_usd"] * result["discount_pct"] / 100
    result["discounted_price_usd"] = result["price_usd"] - result["discount_amount_usd"]
    result["log_price_usd"] = np.log1p(result["price_usd"].clip(lower=0))
    result["log_review_count"] = np.log1p(result["review_count"].clip(lower=0))
    result["category_median_price"] = result["category"].map(category_price_median)
    result["price_to_category_median"] = result["price_usd"] / result["category_median_price"].replace(0, np.nan)
    result["rating_minus_category_mean"] = result["avg_rating"] - result["category"].map(category_rating_mean)
    result["price_band"] = pd.cut(
        result["price_usd"], bins=price_bin_edges, include_lowest=True, duplicates="drop"
    ).astype("string")
    result["age_group"] = pd.cut(
        result["age"], bins=age_bin_edges,
        labels=["18-24", "25-34", "35-44", "45-54", "55-64", "65+"],
        include_lowest=True,
    ).astype("string")

    # Low-cardinality interactions that allow context-dependent effects.
    result["platform_device"] = result["platform"].astype(str) + "__" + result["device_os"].astype(str)
    result["page_slot"] = result["page_type"].astype(str) + "__slot_" + result["slot_position"].astype(str)
    result["category_page"] = result["category"].astype(str) + "__" + result["page_type"].astype(str)
    result["loyalty_platform"] = result["loyalty_tier"].astype(str) + "__" + result["platform"].astype(str)
    result["gender_category"] = result["gender"].astype(str) + "__" + result["category"].astype(str)
    return result

train_fe = add_row_features(train)
validation_fe = add_row_features(validation)
test_fe = add_row_features(test)
print(f"Columns after row-level features: {train_fe.shape[1]}")

Columns after row-level features: 45


## Create leakage-safe historical count and CTR features

For each training impression, cumulative values exclude the current click. Validation and test features use aggregates learned only from training clicks. Smoothing pulls small groups toward the global training CTR.

In [6]:
smoothing_strength = 50
history_specs = {
    "user": ["user_id"],
    "product": ["product_id"],
    "category": ["category"],
    "brand": ["brand"],
    "page": ["page_type"],
    "user_category": ["user_id", "category"],
}
history_tables = {}

for feature_name, keys in history_specs.items():
    grouped = train_fe.groupby(keys, dropna=False, sort=False)[target]
    prior_count = grouped.cumcount()
    prior_clicks = grouped.cumsum() - train_fe[target]
    train_fe[f"{feature_name}_past_impressions"] = prior_count.astype("int32")
    train_fe[f"{feature_name}_past_clicks"] = prior_clicks.astype("int32")
    train_fe[f"{feature_name}_past_ctr"] = (
        prior_clicks + smoothing_strength * global_ctr
    ) / (prior_count + smoothing_strength)

    table = (
        train_fe.groupby(keys, dropna=False)[target]
        .agg(history_impressions="size", history_clicks="sum")
        .reset_index()
    )
    table["history_ctr"] = (
        table["history_clicks"] + smoothing_strength * global_ctr
    ) / (table["history_impressions"] + smoothing_strength)
    history_tables[feature_name] = table

    for later_name, later_frame in [("validation", validation_fe), ("test", test_fe)]:
        merged = later_frame[keys].merge(table, on=keys, how="left", sort=False)
        later_frame[f"{feature_name}_past_impressions"] = merged["history_impressions"].fillna(0).astype("int32").to_numpy()
        later_frame[f"{feature_name}_past_clicks"] = merged["history_clicks"].fillna(0).astype("int32").to_numpy()
        later_frame[f"{feature_name}_past_ctr"] = merged["history_ctr"].fillna(global_ctr).to_numpy()

history_feature_count = len(history_specs) * 3
print(f"Created {history_feature_count} historical features.")

Created 18 historical features.


## Add history-derived affinity and cold-start indicators

In [7]:
for frame in [train_fe, validation_fe, test_fe]:
    frame["user_category_affinity"] = (
        frame["user_category_past_impressions"]
        / frame["user_past_impressions"].replace(0, np.nan)
    ).fillna(0)
    frame["is_new_user"] = frame["user_past_impressions"].eq(0).astype("int8")
    frame["is_new_product"] = frame["product_past_impressions"].eq(0).astype("int8")
    frame["user_product_ctr_gap"] = frame["user_past_ctr"] - frame["product_past_ctr"]
    frame["category_page_ctr_gap"] = frame["category_past_ctr"] - frame["page_past_ctr"]

engineered_splits = {"train": train_fe, "validation": validation_fe, "test": test_fe}
new_features = [column for column in train_fe.columns if column not in train.columns]
print(f"Total new features: {len(new_features)}")
display(pd.DataFrame({"engineered_feature": new_features}))

Total new features: 46


,engineered_feature
0,event_month
1,event_day_of_month
2,event_weekday_number
3,is_weekend
4,time_of_day
5,hour_sin
6,hour_cos
7,days_since_training_start
8,has_discount
9,discount_amount_usd


## Validate engineered datasets and inspect target relationships

In [8]:
validation_rows = []
for name, frame in engineered_splits.items():
    numeric_new = frame[new_features].select_dtypes(include="number")
    validation_rows.append({
        "split": name,
        "rows": len(frame),
        "columns": frame.shape[1],
        "new_features": len(new_features),
        "missing_engineered_values": int(frame[new_features].isna().sum().sum()),
        "infinite_engineered_values": int(np.isinf(numeric_new.to_numpy()).sum()),
        "click_rate": frame[target].mean(),
    })
    if len(frame) != len(splits[name]):
        raise ValueError(f"Feature engineering changed the row count for {name}.")
    if not frame["impression_id"].equals(splits[name].sort_values(["event_ts", "impression_id"])["impression_id"].reset_index(drop=True)):
        raise ValueError(f"Feature engineering changed row identity or order for {name}.")

feature_validation = pd.DataFrame(validation_rows).set_index("split")
display(feature_validation)

numeric_engineered = train_fe[new_features].select_dtypes(include="number").columns
target_correlations = (
    train_fe[list(numeric_engineered) + [target]]
    .corr(numeric_only=True)[target]
    .drop(target)
    .sort_values(key=abs, ascending=False)
    .to_frame("training_correlation_with_clicked")
)
display(target_correlations.head(20))

,rows,columns,new_features,missing_engineered_values,infinite_engineered_values,click_rate
split,,,,,,
train,315000,68,46,0,0,0.0682
validation,67500,68,46,0,0,0.0701
test,67500,68,46,0,0,0.0706


,training_correlation_with_clicked
product_past_ctr,0.2044
user_product_ctr_gap,-0.1860
product_past_clicks,0.1559
category_past_ctr,0.1494
brand_past_ctr,0.1476
category_page_ctr_gap,0.1394
user_category_past_clicks,0.1159
user_category_past_ctr,0.1065
rating_minus_category_mean,0.1063
category_past_clicks,0.1037


## Save engineered datasets and feature metadata

In [9]:
output_paths = {}
for name, frame in engineered_splits.items():
    output_path = engineered_dir / f"{name}_engineered.csv"
    frame.to_csv(output_path, index=False)
    output_paths[name] = output_path
    print(f"Saved {name}: {frame.shape} -> {output_path}")

metadata = {
    "target": target,
    "global_training_ctr": global_ctr,
    "smoothing_strength": smoothing_strength,
    "original_columns": train.columns.tolist(),
    "engineered_features": new_features,
    "history_groups": history_specs,
    "leakage_policy": "Training rows use prior events only; validation and test use training aggregates only.",
    "reference_values": reference_values,
}
with (engineered_dir / "feature_metadata.json").open("w") as file:
    json.dump(metadata, file, indent=2)
joblib.dump(
    {
        "category_price_median": category_price_median,
        "category_rating_mean": category_rating_mean,
        "history_tables": history_tables,
        "reference_values": reference_values,
    },
    engineered_dir / "feature_reference_tables.joblib",
)

Saved train: (315000, 68) -> /Users/namees.local/Desktop/ensamble_technique/data/processed/engineered/train_engineered.csv
Saved validation: (67500, 68) -> /Users/namees.local/Desktop/ensamble_technique/data/processed/engineered/validation_engineered.csv
Saved test: (67500, 68) -> /Users/namees.local/Desktop/ensamble_technique/data/processed/engineered/test_engineered.csv


['/Users/namees.local/Desktop/ensamble_technique/data/processed/engineered/feature_reference_tables.joblib']

## Feature-engineering recap and modeling handoff

In [10]:
top_features = target_correlations.head(5).index.tolist()
missing_total = int(feature_validation["missing_engineered_values"].sum())

display(Markdown(f"""
### What was created

- **{len(new_features)} new features** were added without changing row counts or the `clicked` target.
- Time features describe month, weekday, weekend, time of day, cyclic hour, and elapsed days.
- Product features describe discount value, final price, price relative to category, rating relative to category, and review popularity.
- Interaction features let the model learn combinations such as platform–device, page–slot, category–page, loyalty–platform, and gender–category.
- Historical features describe prior exposure and smoothed prior CTR for users, products, categories, brands, pages, and user–category pairs.
- Cold-start flags identify users and products with no available history. User–category affinity measures the share of a user's past impressions belonging to the current category.
- The strongest simple numeric relationships with `clicked` in training are: **{', '.join(top_features)}**. These are clues only; validation performance decides whether they are useful.
- Engineered features contain **{missing_total:,} missing values** across all splits. The preprocessing pipeline must impute them using training data only.

### Leakage protection

Training history excludes the current click. Validation and test history uses training data only, so their own targets never become inputs. Category statistics and bin boundaries also come from training only.

### Next step

Use the files in `data/processed/engineered` to fit a new encoding pipeline and train baseline and ensemble classifiers. Compare models on validation PR-AUC, ROC-AUC, log loss, and calibration; keep the test set untouched until final selection.
"""))


### What was created

- **46 new features** were added without changing row counts or the `clicked` target.
- Time features describe month, weekday, weekend, time of day, cyclic hour, and elapsed days.
- Product features describe discount value, final price, price relative to category, rating relative to category, and review popularity.
- Interaction features let the model learn combinations such as platform–device, page–slot, category–page, loyalty–platform, and gender–category.
- Historical features describe prior exposure and smoothed prior CTR for users, products, categories, brands, pages, and user–category pairs.
- Cold-start flags identify users and products with no available history. User–category affinity measures the share of a user's past impressions belonging to the current category.
- The strongest simple numeric relationships with `clicked` in training are: **product_past_ctr, user_product_ctr_gap, product_past_clicks, category_past_ctr, brand_past_ctr**. These are clues only; validation performance decides whether they are useful.
- Engineered features contain **0 missing values** across all splits. The preprocessing pipeline must impute them using training data only.

### Leakage protection

Training history excludes the current click. Validation and test history uses training data only, so their own targets never become inputs. Category statistics and bin boundaries also come from training only.

### Next step

Use the files in `data/processed/engineered` to fit a new encoding pipeline and train baseline and ensemble classifiers. Compare models on validation PR-AUC, ROC-AUC, log loss, and calibration; keep the test set untouched until final selection.
